# Fine-Tuning Gemma 2 2B para Apex AI

Este notebook faz fine-tuning do modelo **Gemma 2 2B IT** da Google com dados da plataforma Apex AI.
Usa LoRA (Low-Rank Adaptation) para treinar de forma eficiente em GPU gratuita do Colab.

**Objetivo**: Criar um modelo Gemma que responda como a Apex AI, especializada em arquitetura, construção, BIM, orçamentos e gestão.

**Tempo estimado**: 30-60 minutos com GPU T4 (gratuita do Colab)

## Instruções:
1. Vá em **Runtime → Change runtime type** → Selecione **T4 GPU**
2. Execute cada célula em ordem (Ctrl+F9 para executar tudo)
3. Ao final, o modelo treinado será baixado como ZIP

## 1. Instalar Dependências

In [ ]:
# Instala as bibliotecas necessárias para fine-tuning
!pip install -q transformers==4.44.0 datasets==2.21.0 accelerate==0.33.0 peft==0.12.0 bitsandbytes==0.43.3

# Verifica se GPU está disponível
import torch
print(f"GPU disponível: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memória: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 2. Fazer Upload do Dataset

Faça upload do arquivo `apex_training_vertex.jsonl` quando solicitado.

In [ ]:
from google.colab import files, userdata
import json, os, io

# Tenta carregar o token do Hugging Face dos Secrets do Colab
HF_TOKEN = userdata.get('HF_TOKEN') or os.environ.get('HF_TOKEN') or None
if HF_TOKEN:
    print("✅ Token do Hugging Face encontrado nos Secrets do Colab!")
    # Já está pronto para uso
else:
    print("⚠️ HF_TOKEN não encontrado nos Secrets do Colab.")
    print("📌 Para configurar: clique no ícone 🔑 (Secrets) na barra lateral esquerda")
    print("   Adicione HF_TOKEN = seu token do Hugging Face (https://huggingface.co/settings/tokens)")

print("📤 Faça upload do arquivo apex_training_vertex.jsonl")
print("O arquivo está em: D:\\AI-constr\\apex-ai-copilot-platform\\training_data\\")
print("💡 Alternativa: baixe de https://raw.githubusercontent.com/jedgard70/apex-ai-copilot-platform/main/training_data/apex_training_vertex.jsonl")

try:
    uploaded = files.upload()
    filename = list(uploaded.keys())[0]
except:
    # Fallback: baixar direto do GitHub
    print("⬇️ Upload cancelado. Baixando do GitHub...")
    !wget -q https://raw.githubusercontent.com/jedgard70/apex-ai-copilot-platform/main/training_data/apex_training_vertex.jsonl -O apex_training_vertex.jsonl
    filename = "apex_training_vertex.jsonl"

# Carrega e exibe estatísticas do dataset
with open(filename, "r") as f:
    lines = f.readlines()

dataset = [json.loads(line) for line in lines]
print(f"\n✅ Dataset carregado: {len(dataset)} exemplos")
print(f"📊 Amostra do primeiro exemplo:")
print(json.dumps(dataset[0], indent=2)[:500])

## 3. Carregar Modelo Gemma 2 2B e Tokenizador

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from huggingface_hub import login as hf_login
import os

MODEL_NAME = "google/gemma-2-2b-it"

# ═════════════════════════════════════════════════
# PASSO 1: Autenticar no Hugging Face
# ═════════════════════════════════════════════════
# O Gemma 2 2B é um modelo "gated" — você precisa:
# 1. Aceitar os termos em https://huggingface.co/google/gemma-2-2b-it
# 2. Criar um token de acesso em https://huggingface.co/settings/tokens
# 3. Adicionar o token ao Colab como Secret (🔑) com o nome HF_TOKEN

# Tenta ler o token de múltiplas fontes
HF_TOKEN = None

# Fonte 1: Secrets do Colab (recomendado)
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
    if HF_TOKEN:
        print("✅ Token encontrado nos Secrets do Colab!")
except:
    pass

# Fonte 2: Variável de ambiente
if not HF_TOKEN:
    HF_TOKEN = os.environ.get('HF_TOKEN')

# Fonte 3: Input manual (interativo)
if not HF_TOKEN:
    from getpass import getpass
    print("\n⚠️ Token não encontrado nos Secrets do Colab.")
    print("Cole seu token do Hugging Face (ou pressione Enter para continuar sem autenticação):")
    HF_TOKEN = getpass()
    if not HF_TOKEN:
        print("⚠️ Continuando sem token. Se houver erro de autenticação, execute:")
        print("   huggingface-cli login")
        print("   E cole seu token manualmente.")
        HF_TOKEN = True  # Fallback para token=True

# Faz o login
if isinstance(HF_TOKEN, str):
    hf_login(token=HF_TOKEN, add_to_git_credential=False)
    print(f"✅ Autenticado com token {HF_TOKEN[:8]}...{HF_TOKEN[-4:]}")

# ═════════════════════════════════════════════════
# PASSO 2: Carregar modelo com quantização 4-bit
# ═════════════════════════════════════════════════
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

print(f"\n🔄 Carregando {MODEL_NAME}... (2-3 min na primeira vez)")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=HF_TOKEN)
tokenizer.padding_side = "right"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    token=HF_TOKEN,
    torch_dtype=torch.bfloat16,
)

print(f"✅ Modelo carregado! Parâmetros: {sum(p.numel() for p in model.parameters())/1e9:.2f}B")
print(f"   Device: {model.device}")
print(f"   Pad token: '{tokenizer.pad_token}' (pad_token_id={tokenizer.pad_token_id})")

## 4. Configurar LoRA (Fine-Tuning Eficiente)

LoRA permite fine-tuning com pouquíssima memória ao treinar apenas matrizes de adaptação.

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType

# Configuração LoRA
lora_config = LoraConfig(
    r=8,                    # Rank do LoRA (maior = mais params treináveis)
    lora_alpha=32,          # Escala (32 é padrão para rank 8)
    target_modules=[        # Layers onde LoRA será aplicado
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

# Prepara modelo para treino em quantização 4-bit
model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, lora_config)

# Mostra quantos parâmetros serão treinados
model.print_trainable_parameters()
# Exemplo de output: trainable params: ~8M / total: ~2.2B → só 0.4% treináveis!

## 5. Preparar Dataset no Formato Instruct

In [ ]:
from datasets import Dataset
import json

# Lê o dataset
with open(filename, "r") as f:
    lines = f.readlines()
raw_data = [json.loads(line) for line in lines]

# Converte para formato instruct:
# <start_of_turn>user\n{input}<end_of_turn>\n<start_of_turn>model\n{output}<end_of_turn>
def format_example(example):
    # O campo depende de como o JSONL foi exportado
    user_text = example.get("input") or example.get("messages", [{}])[0].get("content", "")
    model_text = example.get("output") or example.get("messages", [{}])[1].get("content", "")
    return {
        "text": f"<start_of_turn>user\n{user_text}<end_of_turn>\n<start_of_turn>model\n{model_text}<end_of_turn>"
    }

formatted = [format_example(ex) for ex in raw_data]
dataset = Dataset.from_list(formatted)

print(f"✅ Dataset preparado: {len(dataset)} exemplos")
print(f"\n📝 Exemplo formatado:")
print(formatted[0]["text"][:300])

# Split 90/10 treino/validação
split_dataset = dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = split_dataset["train"]
eval_dataset = split_dataset["test"]
print(f"   Treino: {len(train_dataset)} | Validação: {len(eval_dataset)}")

## 6. Configurar TrainingArguments e Iniciar Treinamento

In [ ]:
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

training_args = TrainingArguments(
    output_dir="./gemma-apex-finetuned",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,   # Batch efetivo = 8
    learning_rate=2e-5,
    warmup_steps=10,
    logging_steps=5,
    eval_strategy="steps",
    eval_steps=20,
    save_steps=50,
    save_total_limit=2,
    fp16=True,                        # Ativa mixed precision (essencial na T4)
    bf16=False,
    gradient_checkpointing=True,      # Economiza VRAM
    optim="paged_adamw_8bit",         # Otimizador eficiente para 4-bit
    report_to="none",                 # Não enviar para wandb/tensorboard
    push_to_hub=False,
    remove_unused_columns=False,
    dataloader_pin_memory=False,
    ddp_find_unused_parameters=False,
)

def tokenize_function(examples):
    tokens = tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=512,
    )
    tokens["labels"] = tokens["input_ids"].copy()
    return tokens

tokenized_train = train_dataset.map(tokenize_function, batched=True, remove_columns=["text"])
tokenized_eval = eval_dataset.map(tokenize_function, batched=True, remove_columns=["text"])

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
    data_collator=data_collator,
)

print("🚀 Iniciando treinamento...")
print(f"   Batch efetivo: {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}")
print(f"   Steps por época: ~{len(tokenized_train) // (training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps)}")
print(f"   Total de steps: ~{len(tokenized_train) // (training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps) * training_args.num_train_epochs}")

trainer.train()

## 7. Avaliar e Testar o Modelo Fine-Tunado

In [ ]:
import torch

# Salva o modelo fine-tunado
output_dir = "./gemma-apex-finetuned-final"
model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)
print(f"✅ Modelo salvo em {output_dir}")

# Testa com algumas perguntas
test_prompts = [
    "O que é a Apex AI Copilot Platform?",
    "Como funciona a Apex AI na construção civil?",
    "Quais são os módulos disponíveis na Apex AI?",
]

for prompt in test_prompts:
    inputs = tokenizer(
        f"<start_of_turn>user\n{prompt}<end_of_turn>\n<start_of_turn>model\n",
        return_tensors="pt",
        truncation=True,
        max_length=256,
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=128,
            temperature=0.7,
            do_sample=True,
            top_p=0.95,
        )

    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    # Extrai apenas a resposta do modelo
    answer = response.split("<start_of_turn>model\n")[-1].strip()
    print(f"\n🧑 Usuário: {prompt}")
    print(f"🤖 Apex AI: {answer[:200]}...")
    print("-" * 50)

## 8. Exportar Modelo para Uso no Ollama (Local / Sem API Gemini)

In [ ]:
# Converte os adapters LoRA para um modelo completo fundido
from peft import PeftModel

# Faz merge dos pesos LoRA no modelo base
merged_model = model.merge_and_unload()
print("✅ Modelo fundido (base + LoRA mergeado)")

# Salva em formato compatível com transformers (GGUF será passo extra)
merged_model.save_pretrained("./gemma-apex-merged")
tokenizer.save_pretrained("./gemma-apex-merged")

print("\n📦 Modelo salvo em ./gemma-apex-merged/")
print("\n📋 INSTRUÇÕES PARA USAR SEM GEMINI API (localmente no Ollama):")
print("=" * 60)
print("""
1. No seu computador, instale o Ollama: https://ollama.com

2. Baixe o GGUF do Gemma 2 2B (já disponível no Ollama):
   ollama pull gemma2:2b

3. Crie um Modelfile:
   > cat > Modelfile << 'EOF'
   FROM gemma2:2b
   SYSTEM "Você é a Apex AI Copilot Platform, uma plataforma especializada em construção civil, engenharia e arquitetura. Responda em português."
   PARAMETER temperature 0.7
   PARAMETER top_p 0.95
   EOF

4. Crie o modelo personalizado:
   ollama create apex-ai:latest -f Modelfile

5. Use sem API key:
   ollama run apex-ai "O que é a Apex AI?"

6. Integre na plataforma Apex AI via API:
   POST http://localhost:11434/api/chat
   {
     "model": "apex-ai",
     "messages": [{"role": "user", "content": "O que é a Apex AI?"}]
   }
""")
print("=" * 60)

# ═══════════════════════════════════════════════════════
# PASSO 2: FAZER UPLOAD DO MODELO PARA O HUGGING FACE HUB
# ═══════════════════════════════════════════════════════
print("\n☁️ Fazendo upload do modelo para o Hugging Face Hub...")

HF_USERNAME = "jedgard70"
HF_REPO_NAME = "gemma-2-2b-apex-ai"

if HF_TOKEN:
    try:
        from huggingface_hub import HfApi, create_repo
        
        api = HfApi(token=HF_TOKEN)
        repo_id = f"{HF_USERNAME}/{HF_REPO_NAME}"
        create_repo(repo_id, private=False, exist_ok=True, token=HF_TOKEN)
        print(f"📁 Repositório {repo_id} pronto!")
        
        api.upload_folder(
            folder_path="./gemma-apex-merged",
            repo_id=repo_id,
            token=HF_TOKEN,
        )
        print(f"✅ Upload completo! Modelo em: https://huggingface.co/{repo_id}")
        print(f"\n🌐 Agora seu modelo está acessível online!")
        print(f"   API Inference: https://api-inference.huggingface.co/models/{repo_id}")
        
        # Salva o endpoint para uso posterior
        ENDPOINT_INFO = f"{{\"repo_id\":\"{repo_id}\",\"status\":\"uploaded\"}}"
        with open("apex_model_deploy_info.json", "w") as f:
            f.write(ENDPOINT_INFO)
        print("📄 Informações de deploy salvas em apex_model_deploy_info.json")
        
    except Exception as e:
        print(f"⚠️ Erro no upload: {e}")
        print("💡 Tente manualmente:")
        print(f"   1. Vá em https://huggingface.co/new")
        print(f"   2. Crie repositório: {HF_REPO_NAME}")
        print(f"   3. Faça upload manual da pasta ./gemma-apex-merged/")
else:
    print("\n⚠️ Pulei o upload (HF_TOKEN não configurado).")
    print("📌 Para ativar o upload automático:")
    print("   1. Vá em https://huggingface.co/settings/tokens")
    print("   2. Crie um token com permissão 'write'")
    print("   3. No Colab: 🔑 Secrets → HF_TOKEN = seu_token")
    print("   4. Reexecute esta célula")

# ═══════════════════════════════════════════════════════
# PASSO 3: INSTRUÇÕES PARA USAR NO APEXGLOBALAI.COM
# ═══════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("🌍 COMO USAR NO APEXGLOBALAI.COM (para seus clientes):")
print("=" * 60)
repo_id_safe = f"{HF_USERNAME}/{HF_REPO_NAME}"
print(f"""
Após o upload, seu modelo estará disponível em:
  https://huggingface.co/{repo_id_safe}

Para usar no Apex AI:
  1. Acesse o Owner Console → Deploy 🚀
  2. Clique em "Deploy Inference Endpoint"
  3. O modelo será ativado e aparecerá no seletor de modelos
  4. Seus clientes poderão usar sem API key

Para testar manualmente:
  curl -X POST https://api-inference.huggingface.co/models/{repo_id_safe} \\
    -H "Authorization: Bearer SEU_TOKEN" \\
    -H "Content-Type: application/json" \\
    -d '{{"inputs": "O que é a Apex AI?"}}'
""")

# ═══════════════════════════════════════════════════════
# PASSO 4: BAIXAR O MODELO (para uso local no Ollama)
# ═══════════════════════════════════════════════════════
import shutil
shutil.make_archive("gemma-apex-merged", "zip", "./gemma-apex-merged")
print("📦 Arquivo compactado criado: gemma-apex-merged.zip")
print("   (Faça o download pelo painel do Colab: Files → Download)")
print("\n" + "=" * 60)
print("🎯 🎉 TREINAMENTO CONCLUÍDO! 🎉 🎯")
print("=" * 60)
print(f"""
   ✓ Modelo fine-tunado com {len(train_dataset)} exemplos da Apex AI
   ✓ Upload para Hugging Face: https://huggingface.co/{repo_id_safe}
   ✓ ZIP pronto: gemma-apex-merged.zip (use no Ollama local)

   PRÓXIMO PASSO:
   1. No Apex AI Owner Console → clique em Deploy 🚀
   2. Ative o Inference Endpoint (gratuito serverless)
   3. Pronto! Seus clientes usam o modelo direto do site
""")